# NLP Sentiment Classification

A complete sentiment classification pipeline: loading labeled text data, cleaning it, converting text into numerical features using TF-IDF, training a classifier, and evaluating its performance.

**Dataset:** `Dataset.xlsx` — contains two columns: `text` (the raw text/review) and `sentiment` (the label).


In [ ]:
import pandas as pd
import numpy as np
import re
import string

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("All libraries imported successfully!")

## Step 1: Load the Dataset

In [ ]:
# Load the dataset
data1 = pd.read_excel(r'Dataset.xlsx', sheet_name="Sheet1")

df = pd.DataFrame(data1, columns=['text', 'sentiment'])

print("Dataset Shape:", df.shape)
df.head()

In [ ]:
# Check class distribution
print("Sentiment distribution:")
print(df['sentiment'].value_counts())

print("\nMissing values:")
print(df.isnull().sum())

## Step 2: Clean the Text Data

Before converting text into numbers, we remove noise — punctuation, extra whitespace, and lowercase everything so the model treats `Good` and `good` the same way.

In [ ]:
def clean_text(text):
    """
    Basic text cleaning:
    - Lowercase everything
    - Remove punctuation
    - Remove numbers
    - Remove extra whitespace
    """
    text = str(text).lower()
    text = re.sub(r'[%s]' % re.escape(string.punctuation), '', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Drop missing rows just in case
df = df.dropna(subset=['text', 'sentiment'])

df['clean_text'] = df['text'].apply(clean_text)

print("Before cleaning:")
print(df['text'].iloc[0])
print("\nAfter cleaning:")
print(df['clean_text'].iloc[0])

## Step 3: Encode Sentiment Labels

Machine learning models need numbers, not text labels, so we map each sentiment class to an integer.

In [ ]:
unique_labels = sorted(df['sentiment'].unique())
label_to_num = {label: i for i, label in enumerate(unique_labels)}
num_to_label = {i: label for label, i in label_to_num.items()}

df['label'] = df['sentiment'].map(label_to_num)

print("Label mapping:", label_to_num)
df[['text', 'sentiment', 'label']].head()

## Step 4: Train/Test Split

In [ ]:
X = df['clean_text']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training samples: {len(X_train)}")
print(f"Testing samples : {len(X_test)}")

## Step 5: Convert Text to Features (TF-IDF)

TF-IDF (Term Frequency–Inverse Document Frequency) turns each piece of text into a vector of numbers, weighting words by how important/unique they are to a document compared to the whole dataset.

In [ ]:
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), stop_words='english')

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print(f"TF-IDF training matrix shape: {X_train_tfidf.shape}")
print(f"TF-IDF testing matrix shape : {X_test_tfidf.shape}")

## Step 6: Train the Classifier

We train two models — Naive Bayes (a strong, fast baseline for text classification) and Logistic Regression — and compare them.

In [ ]:
# Model 1: Multinomial Naive Bayes
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)
nb_preds = nb_model.predict(X_test_tfidf)
nb_accuracy = accuracy_score(y_test, nb_preds)

print(f"Naive Bayes Accuracy: {nb_accuracy:.4f}")

In [ ]:
# Model 2: Logistic Regression
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_tfidf, y_train)
lr_preds = lr_model.predict(X_test_tfidf)
lr_accuracy = accuracy_score(y_test, lr_preds)

print(f"Logistic Regression Accuracy: {lr_accuracy:.4f}")

## Step 7: Evaluate the Best Model

In [ ]:
# Pick whichever model performed better
if lr_accuracy >= nb_accuracy:
    best_model_name = "Logistic Regression"
    best_preds = lr_preds
else:
    best_model_name = "Naive Bayes"
    best_preds = nb_preds

print(f"Best Model: {best_model_name}")
print("\nClassification Report:")
print(classification_report(y_test, best_preds, target_names=[str(l) for l in unique_labels]))

In [ ]:
# Confusion matrix heatmap
cm = confusion_matrix(y_test, best_preds)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=unique_labels, yticklabels=unique_labels)
plt.title(f'Confusion Matrix — {best_model_name}', fontsize=13, fontweight='bold')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

## Step 8: Try It on Custom Text

A quick sanity check — predict sentiment on a few new, unseen sentences.

In [ ]:
def predict_sentiment(text, model=lr_model):
    cleaned = clean_text(text)
    vec = vectorizer.transform([cleaned])
    pred_label = model.predict(vec)[0]
    return num_to_label[pred_label]

sample_texts = [
    "I absolutely loved this, it exceeded my expectations!",
    "This was the worst experience I have ever had.",
    "It was okay, nothing special."
]

for text in sample_texts:
    print(f"Text: {text}")
    print(f"Predicted Sentiment: {predict_sentiment(text)}\n")

## Summary

- Cleaned raw text (lowercasing, punctuation/number removal)
- Converted text into numerical features using **TF-IDF** (unigrams + bigrams)
- Trained and compared **Naive Bayes** and **Logistic Regression** classifiers
- Evaluated performance using accuracy, a full classification report, and a confusion matrix
- Tested the final model on new, unseen text samples

## Possible Next Steps
- Try word embeddings (Word2Vec, GloVe) or transformer-based embeddings (BERT) instead of TF-IDF
- Handle class imbalance if one sentiment class dominates the dataset
- Add cross-validation for more robust accuracy estimates
- Deploy the model behind a simple API or Streamlit app for live predictions
